# 第 4 章：从检索结果到可信回答——检索增强生成（RAG, Retrieval-Augmented Generation）进阶

> 读者已经知道基础检索与向量化。本章把问题推向工程现场：候选文档找到了，系统怎样决定“能否回答”、怎样组织证据、何时多检索一步？

**本章成果**：运行一个无需数据库、网络与模型密钥的微型证据管线；读懂本项目的检索、结构化查询语言（SQL, Structured Query Language）执行和回答规则；根据问题形态选择查询改写、多跳检索、层级检索或图检索。

本章的“模拟重排器”和“模板回答器”只展示控制流，不代表模型效果。每种高级方案都应先与简单管线做同一评测集上的对照。

## 4.1 为什么检索命中仍可能答错

RAG 把外部资料送入大语言模型（LLM, Large Language Model）的上下文。它解决知识更新和证据获取问题，但并不自动保证回答正确。例如：

- 用户问“8 月华东净销售额是多少”，文档中有净销售额定义，却没有实际数值。仅靠检索不能报出金额，必须执行查询。
- 检索返回“总销售额”和“净销售额”两张卡片。生成器如果忽略“扣退款”，数字就会偏大。
- 引用了文档标识符（ID, Identifier），也可能引用错段落；**有引用**不等于**引用支持该结论**。

建议把结果看成三类：**有充分证据可回答**、**缺少条件需澄清**、**没有证据或未获授权需拒答**。本项目的智能体（Agent）指令要求先检索业务口径，必要时核对字段，涉及数值时必须调用 SQL 工具并返回实际 SQL 和知识文档 ID。

### 离线准备：直接读取项目的知识卡片

Notebook 从当前目录向上寻找项目根目录；无论在根目录还是 tutorial 目录运行都可找到数据。以下代码只读取 db/knowledge.json，不会连接数据库或外部应用程序编程接口（API, Application Programming Interface）。

In [1]:
from pathlib import Path
import json
import re

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "db" / "knowledge.json").is_file()),
    None,
)
assert ROOT is not None, "请在本项目根目录或 tutorial 目录运行 Notebook"
DOCS = json.loads((ROOT / "db" / "knowledge.json").read_text(encoding="utf-8"))
BY_ID = {doc["id"]: doc for doc in DOCS}
print(f"项目：{ROOT.name}；知识卡片：{len(DOCS)} 张")
print(BY_ID["metric.net_sales"]["content"])

项目：nl2sql-rag；知识卡片：13 张
净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。


## 4.2 构造证据包：来源、顺序与预算

把“相关片段”交给生成器前，应保留稳定来源 ID、内容、权限与版本，并按可解释的顺序放入提示词。生产系统应按模型分词器的词元（token）预算裁剪；下面以字符数演示容量限制。**截断本身可能删掉关键限定条件**，因此先为指标口径和关联规则保留位置，再考虑附加示例。

公式可写为：

$$
\text{可用上下文预算} =
\text{模型窗口} - \text{系统指令} - \text{问题} - \text{预留输出}
$$

本项目在 src/nl2sql/retrieval.py 中拼接“[文档 ID] 内容”，并有教学规模的字符截断。真实业务中应按 token 控制，且不能把权限规则交给模型自行遵守。

In [2]:
def evidence_pack(doc_ids: list[str], max_chars: int = 1500) -> dict:
    parts, kept, used = [], [], 0
    for doc_id in dict.fromkeys(doc_ids):  # 去重且保留顺序
        doc = BY_ID[doc_id]
        part = f"[{doc_id}] {doc['content']}"
        if used + len(part) > max_chars:
            break
        parts.append(part)
        kept.append(doc_id)
        used += len(part)
    return {"doc_ids": kept, "context": "\n\n".join(parts), "chars": used}

bundle = evidence_pack(
    ["metric.net_sales", "table.payments", "table.refunds", "rule.time"],
    max_chars=1500,
)
assert bundle["doc_ids"][0] == "metric.net_sales"
print(bundle["context"])

[metric.net_sales] 净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。

[table.payments] 支付表 analytics.payments：一张订单最多一笔支付；order_id 唯一，amount 为成功支付金额，paid_at 为支付时间，status='SUCCEEDED' 表示成功。按支付日期统计收入应使用 paid_at。

[table.refunds] 退款表 analytics.refunds：一张订单可以有多笔退款；只有 status='APPROVED' 的 amount 才计入退款金额。关联订单时应先按 order_id 聚合退款，避免把一笔支付重复计算。

[rule.time] 时间规则：所有时间列为 timestamptz，样例数据采用 UTC。下单量以 ordered_at 为基准，收入以 paid_at 为基准。月范围使用 >= 月初 AND < 下月月初，避免月底边界错误。


### 生成前的“能否回答”门槛

下面的函数故意不用 LLM：它把“解释指标口径”与“计算实际金额”分开，并要求每个可回答结论附稳定 ID。它不是通用问答器，却能让关键约束看得见。工程上还要核对每个具体断言是否由引用片段支持；仅检查 ID 存在是不够的。

In [3]:
def demo_answer(question: str, evidence_ids: list[str], sql_result: dict | None = None) -> dict:
    ids = set(evidence_ids)
    if "净销售额" not in question:
        return {"status": "abstain", "answer": "此示例只演示净销售额问题。", "citations": []}
    if "metric.net_sales" not in ids:
        return {"status": "abstain", "answer": "缺少净销售额的业务定义，不能推断口径。", "citations": []}
    if any(word in question for word in ("多少", "金额", "数值")):
        if not sql_result or not sql_result.get("ok"):
            return {"status": "abstain", "answer": "已有指标定义，但尚无实际查询结果，不能编造数字。", "citations": ["metric.net_sales"]}
        return {
            "status": "answered",
            "answer": f"查询结果为 {sql_result['value']}；请同时展示时间范围和执行的 SQL。",
            "citations": ["metric.net_sales"],
        }
    return {
        "status": "answered",
        "answer": "净销售额以成功支付金额减去这些订单的已批准退款金额计算。",
        "citations": ["metric.net_sales"],
    }

print(demo_answer("净销售额如何定义？", bundle["doc_ids"]))
print(demo_answer("8 月净销售额是多少？", bundle["doc_ids"]))
assert demo_answer("8 月净销售额是多少？", bundle["doc_ids"])["status"] == "abstain"

{'status': 'answered', 'answer': '净销售额以成功支付金额减去这些订单的已批准退款金额计算。', 'citations': ['metric.net_sales']}
{'status': 'abstain', 'answer': '已有指标定义，但尚无实际查询结果，不能编造数字。', 'citations': ['metric.net_sales']}


## 4.3 查询改写与分解：先解决“用户没说全”

**查询改写**把口语、简称、错别字或多轮追问变为更适合检索的表达。例如“退款后卖了多少”可改为“净销售额”；但改写可能改错原意。**多查询检索**从几个角度检索，再合并候选；它扩大召回，也增加检索成本与噪声。假设文档嵌入（HyDE, Hypothetical Document Embeddings）先生成一篇可能相关的文档再向量化，适合无标注的零样本检索，但生成的“假设文档”可能含虚构细节，**不能当最终证据**。

**查询分解**适合一个问题需要多个不同事实时。例如“华东 8 月净销售额与订单量相比如何？”至少涉及两种指标及各自时间字段；直接把整句送入单一路检索，可能只找回一半。下面的改写是显式规则，以便离线复现；生产可换 LLM 或规则模型，并要保留原始问题做对照。

In [4]:
def rewrite_demo(question: str) -> list[str]:
    variants = [question]
    if "退款后" in question:
        variants.append(question.replace("退款后卖了多少", "净销售额是多少").replace("退款后", "净销售额"))
    if "净销售额" in question:
        variants.extend(["成功支付金额与已批准退款", "净收入"])
    return list(dict.fromkeys(variants))

def decompose_demo(question: str) -> list[str]:
    if "净销售额" in question and "订单量" in question:
        return ["净销售额的口径与相关表", "订单量的口径与相关表", "两个指标各用哪个时间字段"]
    return [question]

q = "华东退款后卖了多少？"
print("改写：", rewrite_demo(q))
print("分解：", decompose_demo("华东 8 月净销售额与订单量相比如何？"))

改写： ['华东退款后卖了多少？', '华东净销售额是多少？']
分解： ['净销售额的口径与相关表', '订单量的口径与相关表', '两个指标各用哪个时间字段']


## 4.4 两阶段排序：广召回，窄精排

第一阶段用关键词、向量或混合检索找出候选。第二阶段的**重排序**（reranking）更仔细地评估“这段内容是否真的回答问题”，常用交叉编码器或托管重排 API。代价是额外模型调用和时延。重排器只会重排已有候选，无法找回第一阶段漏掉的文档；所以先看召回率，再优化前几名的精度。

本项目的运行路径使用向量排序、别名命中和倒数排序融合（RRF, Reciprocal Rank Fusion），然后按知识卡片补齐依赖表。src/nl2sql/rerank.py 是可选实验，**没有进入 Agent 当前的默认路径**。下例只是透明的别名重排玩具，不等于语义重排模型。

In [5]:
def alias_rerank(question: str, candidate_ids: list[str]) -> list[tuple[str, int]]:
    scored = []
    for doc_id in candidate_ids:
        aliases = BY_ID[doc_id].get("aliases", [])
        score = sum(len(alias) for alias in aliases if alias in question)
        scored.append((doc_id, score))
    return sorted(scored, key=lambda x: (-x[1], x[0]))

candidates = ["metric.gross_sales", "metric.net_sales", "table.refunds"]
print(alias_rerank("净销售额要扣退款吗？", candidates))
assert "metric.net_sales" not in [
    doc_id for doc_id, _ in alias_rerank("净销售额要扣退款吗？", ["metric.gross_sales", "table.refunds"])
]  # 漏召回无法靠重排修复

[('metric.net_sales', 4), ('metric.gross_sales', 3), ('table.refunds', 2)]


## 4.5 多跳与智能体式检索：下一步依赖上一步

“净销售额”先定位指标定义，才知道要找 payments、refunds、orders；接着核实字段及关联条件。这是一次**多跳检索**：第二跳取决于第一跳的结果。交替推理和检索的研究表明，复杂多步问题中单次“检索后阅读”常常不足。智能体式检索（Agentic RAG）把“还要不要检索、查哪个源、何时停”交给控制器，但会增加模型调用、时延、循环与错误工具选择风险。

可执行的控制流应设轮数、时限、预算、工具白名单及无证据退出。简单问题仍应允许单次检索直接结束。这里借本项目卡片中的 related_tables 字段，离线复现“指标 → 表”的第二跳。

In [6]:
def expand_related_tables(seed_ids: list[str], max_new: int = 8) -> list[str]:
    selected = list(dict.fromkeys(seed_ids))
    for doc_id in list(selected):
        for table in BY_ID[doc_id].get("related_tables", []):
            table_id = f"table.{table}"
            if table_id in BY_ID and table_id not in selected:
                selected.append(table_id)
                if len(selected) - len(seed_ids) >= max_new:
                    return selected
    return selected

multi_hop = expand_related_tables(["metric.net_sales"])
print("第一跳：metric.net_sales")
print("第二跳：", multi_hop[1:])
assert {"table.payments", "table.refunds", "table.orders"} <= set(multi_hop)

第一跳：metric.net_sales
第二跳： ['table.payments', 'table.refunds', 'table.orders']


## 4.6 层级检索、图检索增强生成与结构化查询

一个 500 页报告的问题“整份报告的主要趋势是什么？”不容易由几个相似小块回答。**层级检索**在索引时构造“原文小块 → 章节摘要 → 文档摘要”，查询时先定位较高层，再回到原文验证细节。RAPTOR（Recursive Abstractive Processing for Tree-Organized Retrieval）是递归聚类与摘要的代表。优点是跨长文档的概览能力；代价是索引时的摘要成本、摘要遗漏与更新传播。

图检索增强生成（GraphRAG, Graph Retrieval-Augmented Generation）进一步提取实体、关系和社区摘要。Microsoft GraphRAG 的 **local search** 适合实体附近的具体问题，**global search** 适合整个语料的主题归纳；global search 的 map-reduce 成本较高。建图并非通用默认方案：实体抽取错误会在图上扩散，且小型 FAQ 或明确代码检索通常不必建图。

本项目有一张更朴素的“指标 → 关联表”依赖图（上一单元格）。它**不是** Microsoft GraphRAG，但说明了为什么结构化依赖关系能补足纯向量相似度。对于需要精确聚合的数值问题，最终应交给结构化查询：自然语言转 SQL（NL2SQL, Natural Language to SQL），再由只读数据库计算；不能让摘要代替 SUM。

### 项目案例：净销售额必须由数据库计算

下面的 SQL 是供阅读的查询草案，不会自动执行。关键是**先按订单聚合退款**，再与每笔支付关联；否则一个订单多笔退款会把支付金额重复计算。示例采用知识卡片约定的“按支付时间选订单，即使退款发生在之后仍扣除”。真实执行应走本项目 src/nl2sql/sql_runner.py 的语法校验、白名单、只读角色和超时控制。

In [7]:
sql_example = """
WITH refund_by_order AS (
    SELECT order_id, SUM(amount) AS refund_amount
    FROM analytics.refunds
    WHERE status = 'APPROVED'
    GROUP BY order_id
)
SELECT COALESCE(SUM(p.amount - COALESCE(r.refund_amount, 0)), 0) AS net_sales
FROM analytics.payments AS p
JOIN analytics.orders AS o ON o.order_id = p.order_id
JOIN analytics.customers AS c ON c.customer_id = o.customer_id
LEFT JOIN refund_by_order AS r ON r.order_id = o.order_id
WHERE p.status = 'SUCCEEDED'
  AND p.paid_at >= TIMESTAMPTZ '2026-08-01 00:00:00+00'
  AND p.paid_at < TIMESTAMPTZ '2026-09-01 00:00:00+00'
  AND c.region = '华东'
""".strip()
print(sql_example)
print("证据：", ["metric.net_sales", "rule.time", "join.customer_orders"])

WITH refund_by_order AS (
    SELECT order_id, SUM(amount) AS refund_amount
    FROM analytics.refunds
    WHERE status = 'APPROVED'
    GROUP BY order_id
)
SELECT COALESCE(SUM(p.amount - COALESCE(r.refund_amount, 0)), 0) AS net_sales
FROM analytics.payments AS p
JOIN analytics.orders AS o ON o.order_id = p.order_id
JOIN analytics.customers AS c ON c.customer_id = o.customer_id
LEFT JOIN refund_by_order AS r ON r.order_id = o.order_id
WHERE p.status = 'SUCCEEDED'
  AND p.paid_at >= TIMESTAMPTZ '2026-08-01 00:00:00+00'
  AND p.paid_at < TIMESTAMPTZ '2026-09-01 00:00:00+00'
  AND c.region = '华东'
证据： ['metric.net_sales', 'rule.time', 'join.customer_orders']


## 4.7 上下文压缩：少送文字，不丢限定

候选越多，提示词越长、成本越高，且有长上下文“中间信息被忽略”的风险。**上下文压缩**可以提取与问题相关的句子、去重相似片段、保留标题和来源。代价是可能误删否定词、时间口径或限定条件。因此压缩前后应做证据覆盖测试；金额、法律和安全规则应优先保留原文。

下面故意做一个危险的子句提取演示：只保留出现“净销售额”的子句，会把“退款发生在窗口之后也要扣除”这样的时间规则删掉。请把它当作反例，而非生产压缩器。

In [8]:
text = BY_ID["metric.net_sales"]["content"]
def naive_compress(text: str, keyword: str) -> str:
    return "；".join(part for part in text.split("；") if keyword in part)

short = naive_compress(text, "净销售额")
print("原文：", text)
print("压缩：", short)
print("是否丢失时间限定：", "即使退款发生在窗口之后" in text and "即使退款发生在窗口之后" not in short)
assert len(short) < len(text)

原文： 净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单；即使退款发生在窗口之后，也从所选订单金额中扣除。
压缩： 净销售额：在选定的成功支付订单集合中，SUM(payments.amount) 减去这些订单的 APPROVED 退款金额。先按 order_id 聚合退款再 LEFT JOIN，避免多笔退款放大支付金额。时间窗口默认以 payments.paid_at 选择订单
是否丢失时间限定： True


## 4.8 如何选择：按问题复杂度升级

| 问题形态 | 首选方案 | 升级条件 | 主要代价 |
|---|---|---|---|
| 明确术语、错误码、表名 | 关键词或混合检索 | 术语有同义写法时加向量 | 索引与融合配置 |
| 单一事实、资料很少 | 简单检索或直接传入上下文 | 证据常漏时调整切块、重排 | 检索时延 |
| 指标涉及多个表 | 检索口径与表依赖，再运行 SQL | 字段不确定时核验模式 | 数据库治理 |
| 一问多事实、后一步依赖前一步 | 受限的多跳或智能体式检索 | 单跳召回不足且评测证明有益 | 轮数、成本、循环 |
| 整库趋势、实体关系 | 层级摘要或 GraphRAG | 主题归纳经常出现 | 索引成本、摘要失真 |

经验顺序：先修数据质量与切块，再修第一阶段召回，之后考虑重排、改写与多跳。任何升级都同时记录证据质量、时延和费用。

## 练习

**基础 1｜拒答边界。** 将 demo_answer 的问题换成“2026 年 8 月净销售额是多少”。说明为什么返回 abstain。**检查标准**：指出 metric.net_sales 只定义口径，数字必须来自实际 SQL 结果；不得把示例 SQL 误当执行结果。

**基础 2｜重排极限。** 从候选列表删掉 metric.net_sales，再运行 alias_rerank。**检查标准**：解释为何精排不能恢复漏召回，并提出扩大候选集或补充别名的改进。

**进阶 1｜多指标分解。** 为“华东 8 月订单量与净销售额”列出至少两条子问题，分别标明指标卡片、日期列及所需表。**提示**：订单量用 ordered_at；收入用 paid_at。**检查标准**：避免把两种时间口径混为一谈，指出订单量默认排除 CANCELLED。

**进阶 2｜上下文压缩回归。** 为 naive_compress 设计一条自动断言，使它一旦删掉“窗口之后仍扣退款”就失败；再提出一种保留整张指标卡片或关键规则的改法。**检查标准**：断言针对具体业务约束，而非仅对比字符长度。

## 参考资料（一手来源）

- Lewis 等，[Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)：RAG 原始论文。
- Anthropic，[Introducing Contextual Retrieval](https://www.anthropic.com/engineering/contextual-retrieval)：切块上下文丢失、混合检索和重排的实验；文中改善幅度只属于其测试条件。
- Weaviate，[Hybrid search](https://docs.weaviate.io/weaviate/concepts/search/hybrid-search) 与 [Reranking](https://docs.weaviate.io/weaviate/concepts/reranking)：融合与二阶段排序。
- Gao 等，[Precise Zero-Shot Dense Retrieval without Relevance Labels](https://arxiv.org/abs/2212.10496)：HyDE。
- Trivedi 等，[Interleaving Retrieval with Chain-of-Thought Reasoning](https://arxiv.org/abs/2212.10509)：交替检索与多步推理。
- Sarthi 等，[RAPTOR](https://arxiv.org/abs/2401.18059)：树形摘要检索。
- Microsoft，[GraphRAG Query Engine](https://microsoft.github.io/graphrag/query/overview/)：local/global 查询差异；[Agentic RAG 工程指南](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/rag/rag-agentic)：成本、轮数及回退。
- Liu 等，[Lost in the Middle](https://arxiv.org/abs/2307.03172)：长上下文位置效应。